# 01 — EDA: Raw Transaction Data

UCI Online Retail II — a UK-based online retailer's transactional records (2009-12-01 → 2011-12-09, ~1M line items). One row per invoice line. No churn label; this notebook explores structure and quality only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_SEED = 42
DATA_DIR = Path('..') / 'data' / 'raw'

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 1. Load Data

In [ ]:
df = pd.read_csv(
    DATA_DIR / 'online_retail_II.csv',
    dtype={'Customer ID': 'Int64'},
    parse_dates=['InvoiceDate'],
    encoding='ISO-8859-1',
)

df = df.rename(columns={
    'Invoice': 'invoice',
    'StockCode': 'stock_code',
    'Description': 'description',
    'Quantity': 'quantity',
    'InvoiceDate': 'invoice_date',
    'Price': 'price',
    'Customer ID': 'customer_id',
    'Country': 'country',
})

print(df.shape)
df.head(5)

In [ ]:
print(f'Rows                : {len(df):,}')
print(f'Unique invoices     : {df["invoice"].nunique():,}')
print(f'Unique stock codes  : {df["stock_code"].nunique():,}')
print(f'Unique customers    : {df["customer_id"].nunique():,} (excluding nulls)')
print(f'Unique countries    : {df["country"].nunique()}')
print(f'Invoice date range  : {df["invoice_date"].min()} → {df["invoice_date"].max()}')


## 2. Schema & Data Types

In [ ]:
# Ger number of entries, N of non-null values and variable types
df.info()

## 3. Missing Values

In [ ]:
missing_pct = df.isnull().mean() * 100
print('Percentage of missing values by variable:')
missing_pct.round(3)

## 4. Data Quality Checks

Things to verify on a raw transactional dump:
- Cancellations (invoices prefixed with `C`) and how they correlate with negative quantities.
- For returns (negative quantity), whether `price` is also signed negative or stays positive.
- Non-positive prices (data errors, adjustments).
- Exact duplicate rows.
- Rows with missing `customer_id` (cannot be attributed to a customer).


**Quick quality flags.** A first pass over the obvious failure modes in one place: how many rows are cancellations (C-prefix), have negative or zero quantity, negative or zero price, a missing `customer_id`, or are exact duplicates. The crosstab then shows how the cancellation flag lines up with negative quantity, and the final block checks whether negative-quantity (return) rows keep a positive `price` or carry a signed-negative one.

In [ ]:
is_cancellation = df['invoice'].astype(str).str.startswith('C')
neg_qty          = (df['quantity'] < 0)
zero_qty         = (df['quantity'] == 0)
neg_price        = (df['price'] < 0)
zero_price       = (df['price'] == 0)
missing_customer = df['customer_id'].isna()
duplicates       = df.duplicated()

print(f'Cancellation invoices (C-prefix)   : {is_cancellation.sum():>8,} ({is_cancellation.mean()*100:.2f}%)')
print(f'Rows with quantity < 0             : {neg_qty.sum():>8,}')
print(f'Rows with quantity == 0            : {zero_qty.sum():>8,}')
print(f'Rows with price < 0                : {neg_price.sum():>8,}')
print(f'Rows with price == 0               : {zero_price.sum():>8,}')
print(f'Rows with missing customer_id      : {missing_customer.sum():>8,} ({missing_customer.mean()*100:.2f}%)')
print(f'Exact duplicate rows               : {duplicates.sum():>8,} ({duplicates.mean()*100:.2f}%)')

print()
print('Overlap: cancellation flag vs negative quantity')
print(pd.crosstab(is_cancellation, neg_qty, rownames=['is_cancellation'], colnames=['quantity<0']))

print()
print('For rows with negative quantity (returns), is price also negative?')
ret = df[neg_qty]
print(f'  Rows with quantity < 0      : {len(ret):,}')
print(f'    price > 0                 : {(ret["price"] > 0).sum():,}')
print(f'    price == 0                : {(ret["price"] == 0).sum():,}')
print(f'    price < 0                 : {(ret["price"] < 0).sum():,}')
print()
print('Price summary on negative-quantity rows:')
print(ret['price'].describe().round(2))

**One stock code → many descriptions?** Count how many distinct (normalised) descriptions each `stock_code` carries. Codes with more than one description signal inconsistent or dirty product labelling.

In [ ]:
desc_norm = df['description'].astype(str).str.strip().str.upper()

code_description = (
    df.assign(description_norm=desc_norm)
      .groupby('stock_code')['description_norm']
      .nunique()
      .sort_values(ascending=False)
      .reset_index(name='unique_descriptions')
)

n_codes    = len(code_description)
n_multi    = (code_description['unique_descriptions'] > 1).sum()
n_nan_desc = df[df['description'].isna()]['stock_code'].nunique()

print(f'Stock codes total                       : {n_codes:,}')
print(f'Stock codes with >1 description         : {n_multi:,} ({n_multi/n_codes*100:.2f}%)')
print(f'Stock codes with any NaN description    : {n_nan_desc:,}')

code_description[code_description['unique_descriptions'] > 1]


In [ ]:
# Check a couple of examples with the stock_ids that have multiple descriptions
for i in ('20713', '21181', '22423', '23084'):
    print(f'Number of unique descriptions for {i}: \n {df[df['stock_code'] == i]['description'].unique()}')

**One description → many stock codes?** The reverse view: how many distinct `stock_code`s share the same (normalised) description. Generic descriptions such as `CHECK`, `DAMAGED` or `?` map to many codes and flag non-product / adjustment rows.

In [ ]:
mask = df['description'].notna()

description_codes = (
    df[mask]
      .assign(description_norm=desc_norm[mask])
      .groupby('description_norm')['stock_code']
      .nunique()
      .sort_values(ascending=False)
      .reset_index(name='unique_stock_codes')
)

n_desc  = len(description_codes)
n_multi = (description_codes['unique_stock_codes'] > 1).sum()

print(f'Distinct descriptions                   : {n_desc:,}')
print(f'Descriptions mapped to >1 stock_code    : {n_multi:,} ({n_multi/n_desc*100:.2f}%)')

description_codes[description_codes['unique_stock_codes'] > 1].head(20)


**Stock code format composition.** Break unique `stock_code`s down by character makeup (numbers only, letters only, mixed alphanumeric, other) to understand the coding scheme and spot non-standard codes (e.g. `POST`, `BANK CHARGES`, `gift_0001_*`).

In [ ]:
# stock_code format breakdown by character composition (unique values only).
unique_codes = pd.Series(df['stock_code'].astype(str).unique())

is_alpha_only   = unique_codes.str.fullmatch(r'[A-Za-z]+')
is_numeric_only = unique_codes.str.fullmatch(r'[0-9]+')
is_alnum_mixed  = unique_codes.str.fullmatch(r'(?=.*[A-Za-z])(?=.*[0-9])[A-Za-z0-9]+')
is_other        = ~(is_alpha_only | is_numeric_only | is_alnum_mixed)

n = len(unique_codes)
print(f'Unique stock codes                  : {n:,}')
print(f'  Numbers only                      : {is_numeric_only.sum():>6,} ({is_numeric_only.mean()*100:.2f}%)')
print(f'  Letters only                      : {is_alpha_only.sum():>6,} ({is_alpha_only.mean()*100:.2f}%)')
print(f'  Letters + numbers (mixed alnum)   : {is_alnum_mixed.sum():>6,} ({is_alnum_mixed.mean()*100:.2f}%)')
print(f'  Other (contains symbols/punct.)   : {is_other.sum():>6,} ({is_other.mean()*100:.2f}%)')
print()
print('Samples — numbers only      :', unique_codes[is_numeric_only].head(10).tolist())
print('Samples — letters only      :', unique_codes[is_alpha_only].head(10).tolist())
print('Samples — letters + numbers :', unique_codes[is_alnum_mixed].head(10).tolist())
print('Samples — other             :', unique_codes[is_other].head(10).tolist())


**Example.** Inspect a few mixed-alphanumeric codes to see what the trailing letter encodes (e.g. `84031A`/`84031B`, `48173C`/`48173A`).

In [ ]:
# Check examples for letters + numbers
stock_code_test1 = '84031A'
stock_code_test2 = '84031B'
print(f'Check the results for {stock_code_test1} and {stock_code_test2}')
print(df[(df['stock_code'] == stock_code_test1) |
         (df['stock_code'] == stock_code_test2)][['stock_code', 'description']].drop_duplicates())

print('\n')

stock_code_test3 = '48173C'
stock_code_test4 = '48173A'
print(f'Check the results for {stock_code_test3} and {stock_code_test4}')
print(df[(df['stock_code'] == stock_code_test3) |
         (df['stock_code'] == stock_code_test4)][['stock_code', 'description']].drop_duplicates())

The cases when the stock_code has same number but different letts is mostly because of a feature that is different in the product (e.g., color).

**Invoice format composition.** The same character-makeup breakdown for `invoice` values, confirming invoices are numeric except for the `C`-prefixed cancellations (which show up as mixed alphanumeric).

In [ ]:
# invoice format breakdown by character composition (unique values only).
unique_invoices = pd.Series(df['invoice'].astype(str).unique())

is_alpha_only   = unique_invoices.str.fullmatch(r'[A-Za-z]+')
is_numeric_only = unique_invoices.str.fullmatch(r'[0-9]+')
is_alnum_mixed  = unique_invoices.str.fullmatch(r'(?=.*[A-Za-z])(?=.*[0-9])[A-Za-z0-9]+')
is_other        = ~(is_alpha_only | is_numeric_only | is_alnum_mixed)

n = len(unique_invoices)
print(f'Unique invoices                     : {n:,}')
print(f'  Numbers only                      : {is_numeric_only.sum():>6,} ({is_numeric_only.mean()*100:.2f}%)')
print(f'  Letters only                      : {is_alpha_only.sum():>6,} ({is_alpha_only.mean()*100:.2f}%)')
print(f'  Letters + numbers (mixed alnum)   : {is_alnum_mixed.sum():>6,} ({is_alnum_mixed.mean()*100:.2f}%)')
print(f'  Other (contains symbols/punct.)   : {is_other.sum():>6,} ({is_other.mean()*100:.2f}%)')
print()
print('Samples — numbers only      :', unique_codes[is_numeric_only].head(10).tolist())
print('Samples — letters only      :', unique_invoices[is_alpha_only].head(10).tolist())
print('Samples — letters + numbers :', unique_invoices[is_alnum_mixed].head(10).tolist())
print('Samples — other             :', unique_invoices[is_other].head(10).tolist())


**Sanity check: one invoice, one customer.** Verify each `invoice` maps to at most one `customer_id`, so invoice-level aggregations downstream can't accidentally blend customers.

In [ ]:
# Sanity check: each invoice should belong to exactly one customer.
# Count unique customer_ids per invoice (NaN customer_ids excluded by nunique by default).
customers_per_invoice = df.groupby('invoice')['customer_id'].nunique()

print(f'Unique invoices                              : {len(customers_per_invoice):,}')
print(f'Invoices with 0 unique customer_ids (all NaN): {(customers_per_invoice == 0).sum():,}')
print(f'Invoices with exactly 1 unique customer_id   : {(customers_per_invoice == 1).sum():,}')
print(f'Invoices with >1 unique customer_id          : {(customers_per_invoice > 1).sum():,}')



## 5. Cancellations

**A cancellation is any row whose `invoice` starts with `C`.** That prefix is the retailer's own explicit marker, so we define cancellations from it rather than inferring them from negative `quantity`.

The C-prefix is the better determiner. From the data-quality crosstab above, of the 22,950 negative-quantity rows only 19,493 are C-invoices — the other 3,457 are *not* cancellations (they are zero-priced stock adjustments / write-offs). Conversely, one C-invoice has a non-negative quantity. So the sign of `quantity` both over-counts (adjustments) and under-counts (the lone non-negative cancellation), whereas the `C` prefix matches the retailer's intent exactly.

Cancellations are a sizeable share of the data and need to be treated explicitly downstream (e.g. excluded, or paired with the original purchase to compute net revenue). Here we just characterise them.

In [ ]:
# Cancellations are defined by the C invoice prefix, NOT by negative quantity.
is_cancellation = df['invoice'].astype(str).str.startswith('C')

cancel_df = df[is_cancellation]
regular_df = df[~is_cancellation]

print(f'Cancellation rows         : {len(cancel_df):,}')
print(f'Regular rows              : {len(regular_df):,}')
print(f'Cancellation unique invoices : {cancel_df["invoice"].nunique():,}')
print(f'Regular unique invoices      : {regular_df["invoice"].nunique():,}')
print()
print('Quantity summary on cancellations:')
print(cancel_df['quantity'].describe().round(2))

# Why C-prefix over negative quantity: the two rules disagree on 3,458 rows.
neg_qty = df['quantity'] < 0
print()
print('C-prefix vs negative-quantity as a cancellation rule:')
print(f'  Cancellations (C-prefix)              : {is_cancellation.sum():,}')
print(f'  Negative-quantity rows                : {neg_qty.sum():,}')
print(f'  Negative quantity but NOT a C-invoice : {(neg_qty & ~is_cancellation).sum():,}  (adjustments/write-offs, not cancellations)')
print(f'  C-invoice but quantity >= 0           : {(is_cancellation & ~neg_qty).sum():,}  (would be missed by the negative-quantity rule)')


In [ ]:
N = 15

# Most-canceled items (by units canceled — flip sign so the number reads positive)
top_canceled = (cancel_df
    .groupby(['stock_code', 'description'])
    .agg(units_canceled=('quantity', lambda x: -x.sum()),
         n_cancel_invoices=('invoice', 'nunique'))
    .sort_values('units_canceled', ascending=False)
    .head(N)
    .reset_index())

# Most-popular items (by units sold, regular invoices only, positive quantity)
top_popular = (regular_df[regular_df['quantity'] > 0]
    .groupby(['stock_code', 'description'])
    .agg(units_sold=('quantity', 'sum'),
         n_invoices=('invoice', 'nunique'))
    .sort_values('units_sold', ascending=False)
    .head(N)
    .reset_index())

print(f'Top {N} most-canceled items (by units canceled):')
print(top_canceled.to_string(index=False))
print()
print(f'Top {N} most-popular items (by units sold, excluding cancellations):')
print(top_popular.to_string(index=False))

overlap = set(top_canceled['stock_code']) & set(top_popular['stock_code'])
print()
print(f'Overlap between top-{N} canceled and top-{N} popular: {len(overlap)} of {N} stock codes')
if overlap:
    overlap_view = (top_popular[top_popular['stock_code'].isin(overlap)]
        .merge(top_canceled[['stock_code', 'units_canceled', 'n_cancel_invoices']],
               on='stock_code', how='left'))
    overlap_view['cancel_to_sold_ratio_pct'] = (
        overlap_view['units_canceled'] / overlap_view['units_sold'] * 100
    ).round(2)
    print()
    print('Shared items (popular AND heavily canceled):')
    print(overlap_view.to_string(index=False))


## 6. Quantity & Price Distributions

Both columns are extremely long-tailed (a handful of bulk orders / high-value items dominate). Histograms below show actual values (no log transform) clipped at the 99th percentile so the shape is readable; raw percentile values are in the description table underneath.


In [ ]:
clean = df[(df['quantity'] > 0) & (df['price'] > 0)].copy()
clean['line_total'] = clean['quantity'] * clean['price']

cols  = ['quantity', 'price', 'line_total']
units = {'quantity': '', 'price': ' (GBP)', 'line_total': ' (GBP)'}
pct_marks = [(50, 'green'), (90, 'orange'), (99, 'red')]  # percentile lines on the histograms

fig, axes = plt.subplots(len(cols), 2, figsize=(15, 4 * len(cols)),
                         gridspec_kw={'width_ratios': [3, 1]})

for i, col in enumerate(cols):
    s = clean[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr            # Tukey rule (lower fence is < 0 here, so no low outliers)
    lower_fence = q1 - 1.5 * iqr
    is_out = (s < lower_fence) | (s > upper_fence)
    n_out, pct_out = is_out.sum(), is_out.mean() * 100
    clip = s.quantile(0.99)

    # --- left: histogram (clipped at 99th pct for readability) with percentile + fence lines
    ax_h = axes[i, 0]
    ax_h.hist(s.clip(upper=clip), bins=60, color='steelblue', alpha=0.7)
    for p, c in pct_marks:
        v = s.quantile(p / 100)
        if v <= clip:
            ax_h.axvline(v, color=c, ls='--', lw=1.2, label=f'{p}th pct = {v:,.2f}')
    if upper_fence <= clip:
        ax_h.axvline(upper_fence, color='black', ls=':', lw=1.6,
                     label=f'Tukey upper fence = {upper_fence:,.2f}')
    else:
        ax_h.plot([], [], color='black', ls=':', lw=1.6,
                  label=f'Tukey upper fence = {upper_fence:,.2f} (beyond view)')
    ax_h.set_title(f'{col}{units[col]} — distribution (clipped at 99th pct)')
    ax_h.set_xlabel(col)
    ax_h.set_ylabel('rows')
    ax_h.legend(fontsize=8)
    ax_h.ticklabel_format(style='plain', axis='both', useOffset=False)
    ax_h.yaxis.get_major_formatter().set_scientific(False)

    # --- right: boxplot (log scale; Tukey 1.5*IQR whiskers, outliers as points)
    ax_b = axes[i, 1]
    ax_b.boxplot(s, whis=1.5, showfliers=True,
                 flierprops=dict(marker='o', markersize=2, alpha=0.25, markerfacecolor='tomato',
                                 markeredgecolor='none'))
    ax_b.set_yscale('log')
    ax_b.set_title(f'{col} — boxplot (log y)')
    ax_b.set_xticks([])
    ax_b.annotate(f'{n_out:,} outliers ({pct_out:.1f}%)\nabove {upper_fence:,.0f}',
                  xy=(0.5, 0.98), xycoords='axes fraction', ha='center', va='top',
                  fontsize=8, color='dimgray')

plt.tight_layout()
plt.show()

print('Summary (positive rows only):')
print(clean[['quantity', 'price', 'line_total']]
      .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(2))


## 7. Country Breakdown

In [ ]:
country_rows      = df['country'].value_counts()
country_customers = df.groupby('country')['customer_id'].nunique()
country_revenue   = (clean.groupby('country')['line_total'].sum().sort_values(ascending=False))

top = pd.DataFrame({
    'rows': country_rows,
    'rows_pct': (country_rows / country_rows.sum() * 100).round(2),
    'unique_customers': country_customers,
    'revenue_gbp': country_revenue.round(0),
}).fillna(0).sort_values('rows', ascending=False).head(15)
top


## 8. Top Products

In [ ]:
top_products = (
    clean.groupby(['stock_code', 'description'])
    .agg(units_sold=('quantity', 'sum'),
         n_customers=('customer_id', 'nunique'),
         n_invoices=('invoice', 'nunique'),
         revenue=('line_total', 'sum'))
    .sort_values('units_sold', ascending=False)
    .head(15)
    .reset_index()
)
print('Top 15 stock codes by units sold (with unique customer counts):')
top_products


## 9. Invoice-Level Aggregations

Each invoice contains multiple lines. Aggregate to one row per invoice and look at basket size and value. **All invoices are included** — cancellations and returns too — so a popular item that's constantly being returned still shows up here.


In [ ]:
df['line_total'] = df['quantity'] * df['price']

invoice_agg = df.groupby('invoice').agg(
    n_lines=('stock_code', 'size'),
    n_unique_products=('stock_code', 'nunique'),
    total_quantity=('quantity', 'sum'),
    total_value=('line_total', 'sum'),
    invoice_date=('invoice_date', 'first'),
    customer_id=('customer_id', 'first'),
    country=('country', 'first'),
)
invoice_agg['is_cancellation'] = invoice_agg.index.astype(str).str.startswith('C')

print(f'Invoices : {len(invoice_agg):,}')
print(f'  Cancellation invoices : {invoice_agg["is_cancellation"].sum():,}')
print(f'  Regular invoices      : {(~invoice_agg["is_cancellation"]).sum():,}')
print()
print('Per-invoice summary (all invoices, includes returns):')
print(invoice_agg[['n_lines', 'n_unique_products', 'total_quantity', 'total_value']]
      .describe(percentiles=[0.5, 0.9, 0.99]).round(2))


## 10. Temporal Patterns

In [ ]:
df['line_total'] = df['quantity'] * df['price']

monthly = (df.assign(month=df['invoice_date'].dt.to_period('M'))
             .groupby('month')
             .agg(transactions=('invoice', 'nunique'),
                  revenue=('line_total', 'sum')))
monthly.index = monthly.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(monthly.index, monthly['transactions'], marker='o')
axes[0].set_title('Unique invoices per month (includes cancellations)')
axes[0].set_ylabel('Invoices')

axes[1].plot(monthly.index, monthly['revenue'], marker='o', color='tab:green')
axes[1].set_title('Net revenue per month (GBP, includes returns/cancellations)')
axes[1].set_ylabel('Revenue (GBP)')

for ax in axes:
    ax.ticklabel_format(style='plain', axis='y', useOffset=False)
    ax.yaxis.get_major_formatter().set_scientific(False)

plt.tight_layout()
plt.show()

monthly.round(0)


## 11. Customer-Level Transaction Counts & Retention

For each attributed customer (excluding rows with missing `customer_id`), what is the percentage who reach an Nth purchase (N = 1..30), and what is the average/median gap in days between the (N-1)th and Nth purchase? A "purchase" here is a **unique invoice** ordered by date (cancellation invoices still count as transactions).


In [ ]:
df['line_total'] = df['quantity'] * df['price']
attributed = df.dropna(subset=['customer_id']).copy()

per_customer = attributed.groupby('customer_id').agg(
    n_invoices=('invoice', 'nunique'),
    n_lines=('stock_code', 'size'),
    total_spend=('line_total', 'sum'),
    first_purchase=('invoice_date', 'min'),
    last_purchase=('invoice_date', 'max'),
)
per_customer['active_days'] = (per_customer['last_purchase'] - per_customer['first_purchase']).dt.days

total_customers = len(per_customer)
print(f'Attributed customers : {total_customers:,}')
print()
print('Per-customer summary:')
print(per_customer[['n_invoices', 'n_lines', 'total_spend', 'active_days']]
      .describe(percentiles=[0.5, 0.9, 0.99]).round(2))

invoice_seq = (attributed
               .groupby(['customer_id', 'invoice'])['invoice_date']
               .min()
               .reset_index()
               .sort_values(['customer_id', 'invoice_date']))
invoice_seq['purchase_rank'] = invoice_seq.groupby('customer_id').cumcount() + 1
invoice_seq['gap_days'] = (invoice_seq
                           .groupby('customer_id')['invoice_date']
                           .diff()
                           .dt.total_seconds() / 86400)

max_n = 30
rows = []
for n in range(1, max_n + 1):
    reached = (per_customer['n_invoices'] >= n).sum()
    pct = reached / total_customers * 100
    if n >= 2:
        gap_rows = invoice_seq.loc[invoice_seq['purchase_rank'] == n, 'gap_days']
        mean_gap = round(gap_rows.mean(), 2) if len(gap_rows) else None
        median_gap = round(gap_rows.median(), 2) if len(gap_rows) else None
        transition = f'{n-1} -> {n}'
    else:
        mean_gap = None
        median_gap = None
        transition = '—'
    rows.append({
        'nth_purchase': n,
        'transition': transition,
        'customers_reaching': reached,
        'pct_of_total': round(pct, 2),
        'mean_days': mean_gap,
        'median_days': median_gap,
    })

retention = pd.DataFrame(rows)
print()
print('Retention curve & inter-purchase gaps (Nth purchase, N = 1..30):')
retention


In [ ]:
# Churn definition: a customer is "churn" if they do NOT make a 2nd purchase
# within N days of their 1st purchase. Customers whose 1st purchase is within
# the last N days of the dataset are excluded (right-censored — they have not
# had a full N-day window in which a 2nd purchase could occur).
dataset_end = df['invoice_date'].max()

first_purchase  = invoice_seq.loc[invoice_seq['purchase_rank'] == 1].set_index('customer_id')['invoice_date']
second_purchase = invoice_seq.loc[invoice_seq['purchase_rank'] == 2].set_index('customer_id')['invoice_date']

customer_purchases = pd.DataFrame({'first': first_purchase, 'second': second_purchase})
customer_purchases['days_to_second'] = (
    (customer_purchases['second'] - customer_purchases['first']).dt.total_seconds() / 86400
)

rows = []
for thresh in [30, 60, 90]:
    cutoff = dataset_end - pd.Timedelta(days=thresh)
    eligible = customer_purchases[customer_purchases['first'] <= cutoff]
    excluded = len(customer_purchases) - len(eligible)

    has_second_in_window = (eligible['days_to_second'] <= thresh).fillna(False)
    churned = (~has_second_in_window).sum()
    non_churned = has_second_in_window.sum()
    pct_churn = churned / len(eligible) * 100

    rows.append({
        'threshold_days': thresh,
        'eligible_customers': len(eligible),
        'excluded_right_censored': excluded,
        'churned': churned,
        'non_churned': non_churned,
        'pct_churned': round(pct_churn, 2),
        'pct_non_churned': round(100 - pct_churn, 2),
    })

churn_by_threshold = pd.DataFrame(rows)
print(f'Dataset end                : {dataset_end}')
print(f'Total attributed customers : {total_customers:,}')
print()
print('Churn % by 1st → 2nd-purchase window (churn = no 2nd purchase within N days of 1st):')
churn_by_threshold


## 12. Inter-Purchase Gaps

For each customer with ≥ 2 invoices, days between consecutive invoices. Complementary view to the per-transition table in section 11 — shows the overall shape of the gap distribution.


In [ ]:
invoice_dates = (attributed
                 .groupby(['customer_id', 'invoice'])['invoice_date']
                 .min()
                 .reset_index()
                 .sort_values(['customer_id', 'invoice_date']))

invoice_dates['gap_days'] = (invoice_dates
                             .groupby('customer_id')['invoice_date']
                             .diff()
                             .dt.total_seconds() / 86400)

gaps = invoice_dates['gap_days'].dropna()
print(f'Customers with ≥ 2 invoices : {(per_customer["n_invoices"] >= 2).sum():,}')
print(f'Inter-invoice gap observations: {len(gaps):,}')
print()
print(gaps.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gaps.clip(upper=365), bins=60)
ax.set_xlabel('Days between consecutive invoices (clipped at 365)')
ax.set_ylabel('Observations')
ax.set_title('Inter-purchase gap distribution')
ax.ticklabel_format(style='plain', axis='y', useOffset=False)
ax.yaxis.get_major_formatter().set_scientific(False)
plt.tight_layout()
plt.show()


Based on the analysis, I am going to predict if the customer will make another purchase within the next 90 days after the 1st purchase. <br> More specifically, if a customer made a 2nd purchase within 90 days after the 1st purchase, then the customer will be assigned 0 for churn. <br> If a customer did not make a purchase after the 1st one, or if the customer made the 2nd purchase after 90 days from the 1st purchase, then the customer will be assigned 1 for churn. <br> This labelled first-transaction dataset is produced by `data/processing/prepare_churn_data.py`.

## Data Quality Issues to Address

The EDA surfaced the following issues that need to be addressed:

1. Missing values for the `customer_id` and `description` variables.
2. For some products, `price = 0`.
3. For one stock code we have more than one description.
4. For the exact same description, we have more than one `stock_code`.
5. The `stock_code` format is inconsistent — codes are stored in different formats: numbers only, letters only, number + letters, or other. The letters-only (e.g., `POST`, `M`, `DOT`) and other (e.g., `gift_0001_80`, `BANK CHARGES`) formats are not valid product codes and require processing.
6. Invoices that start with `C` (cancellations).
7. Outlier values in the continuous variables.

Since I am taking only the 1st transaction and trying to predict if the customer is going to churn in the next 90-days, I will filter to the 1st transaction only and then address the data issues identified as some of those may be fixed after taking the 1st transaction only.